# Explainability-Guided SegFormer-B0 — Full-Scale Training (Phase 2)

Person 2 (Kalana) GPU run. Upload **only** `segformer_full_scale.zip` (built by `make_colab_zip.py`) — not the whole repo.

The zip already contains this code, Person 3's `attention_consistency` package, Person 4's metric modules, and the 5,108 image/mask pairs.

Split: **3576 / 766 / 766**, seed 42 (same held-out test set as Chanupa's U-Net). Epochs: **20**. Variants: `vanilla` and `att`.

**Before running:** Runtime → Change runtime type → GPU (T4 or better). Upload `segformer_full_scale.zip` to Google Drive (`MyDrive/`).


## Step 0: Unzip the Colab bundle

1. Upload `Phase2/Kalana-Person2/segformer_full_scale.zip` to Drive as `MyDrive/segformer_full_scale.zip`.
2. Cells below mount Drive, unzip to `/content/segformer_full_scale` (fast local disk), and write checkpoints/results to `MyDrive/segformer_full_scale_outputs` so a dropped runtime does not lose the run.
3. If the zip is in a different Drive folder, change `ZIP_ON_DRIVE`.


In [ ]:
import sys, zipfile, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

# Upload the zip to Drive, then point this at it:
ZIP_ON_DRIVE = Path("/content/drive/MyDrive/segformer_full_scale.zip")
BUNDLE = Path("/content/segformer_full_scale")
OUTPUTS = Path("/content/drive/MyDrive/segformer_full_scale_outputs")

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Colab: Drive mounted")

    if not (BUNDLE / "paths.py").exists():
        if not ZIP_ON_DRIVE.is_file():
            raise FileNotFoundError(
                f"Upload the zip to {ZIP_ON_DRIVE} (or change ZIP_ON_DRIVE)."
            )
        print("Unzipping", ZIP_ON_DRIVE, "->", BUNDLE.parent, "…")
        with zipfile.ZipFile(ZIP_ON_DRIVE, "r") as z:
            z.extractall(BUNDLE.parent)
        if not (BUNDLE / "paths.py").exists():
            cands = [p for p in BUNDLE.parent.iterdir() if p.is_dir() and (p / "paths.py").exists()]
            if not cands:
                raise FileNotFoundError("Unzipped files, but paths.py was not found.")
            if BUNDLE.exists():
                shutil.rmtree(BUNDLE)
            shutil.move(str(cands[0]), str(BUNDLE))
        print("Unzip done")
    else:
        print("Bundle already unpacked at", BUNDLE)
    HERE = BUNDLE
else:
    HERE = Path.cwd()
    if not (HERE / "paths.py").exists():
        candidate = Path("Phase2/Kalana-Person2").resolve()
        if (candidate / "paths.py").exists():
            HERE = candidate
    OUTPUTS = HERE
    print("Local run: no unzip")

sys.path.insert(0, str(HERE))
print("HERE =", HERE, "-> exists:", HERE.is_dir())


In [ ]:
import subprocess, sys
pkgs = ["transformers", "accelerate", "thop", "tqdm", "opencv-python-headless"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])
print("deps ready")


In [ ]:
import paths
from paths import (
    DATA_IMG_DIR, DATA_MASK_DIR, PERSON3_DIR, PERSON4_DIR, LAYOUT_MODE,
    N_TRAIN, N_VAL, N_TEST, SEED, EPOCHS,
    add_teammate_paths, apply_data_dirs, set_output_dirs,
)

set_output_dirs(OUTPUTS / "checkpoints", OUTPUTS / "results")
add_teammate_paths()
apply_data_dirs()

print("layout:          ", LAYOUT_MODE)
print("Person 3 package:", PERSON3_DIR, "->", (PERSON3_DIR / "attention_consistency").is_dir())
print("Person 4 metrics:", PERSON4_DIR, "->", (PERSON4_DIR / "metrics.py").is_file())
print("images:          ", paths.DATA_IMG_DIR, "->", paths.DATA_IMG_DIR.is_dir())
print("masks:           ", paths.DATA_MASK_DIR, "->", paths.DATA_MASK_DIR.is_dir())
print("checkpoints out: ", paths.CKPT_DIR)
print("results out:     ", paths.RESULTS_DIR)
print(f"split {N_TRAIN}/{N_VAL}/{N_TEST}  seed={SEED}  epochs={EPOCHS}")
assert (PERSON3_DIR / "attention_consistency").is_dir(), "zip missing vendor/attention_consistency"
assert paths.DATA_MASK_DIR.is_dir(), "zip missing data/masks"
assert paths.DATA_IMG_DIR.is_dir(), "zip missing data/images"


## Step 1: Device check


In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
if not torch.cuda.is_available():
    print("WARNING: no GPU. Full-scale 20-epoch x 2 variants will not finish in a reasonable time.")


## Step 2: Split preflight

Confirms the 3576/766/766 seed-42 lists match Chanupa's U-Net split algorithm before any GPU time is spent.


In [ ]:
import runpy
runpy.run_path(str(HERE / "tests" / "test_split_identity.py"), run_name="__main__")


## Step 3: Full-scale training — both variants

Checkpoints write to `OUTPUTS` on Drive, not into anyone else's folder.

Resume: if `segformer_b0_{variant}_best.pt` already exists, that variant is skipped.


In [ ]:
import train_full_scale as T
import paths
from paths import BATCH_SIZE, LR, LAMBDA2, SIGMA, ATT_MODE, SEED, EPOCHS, N_TRAIN, N_VAL, N_TEST

class Args:
    n_train, n_val, n_test = N_TRAIN, N_VAL, N_TEST
    epochs = EPOCHS
    batch_size = BATCH_SIZE
    lr = LR
    lambda2 = LAMBDA2
    sigma = SIGMA
    att_mode = ATT_MODE
    seed = SEED

args = Args()
torch.manual_seed(args.seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(args.seed)

for variant in ("vanilla", "att"):
    best = paths.CKPT_DIR / f"segformer_b0_{variant}_best.pt"
    if best.exists():
        print(f"SKIP train {variant}: {best.name} already exists")
        continue
    T.train_variant(variant, args)


## Step 4: Evaluate both checkpoints (Dice / IoU / F1 / AAMO / efficiency)

Same held-out 766 images as Step 3. Person 4 formulas. Writes only to `OUTPUTS/results` on Drive.


In [ ]:
import eval_full_scale as E

class EvalArgs:
    n_train, n_val, n_test = N_TRAIN, N_VAL, N_TEST
    seed = SEED

rows = []
for variant in ("vanilla", "att"):
    rows.append(E.evaluate_variant(variant, EvalArgs()))
print("\nFull-scale rows:")
for r in rows:
    print(f"  {r['model']}: dice={r['dice']} iou={r['iou']} aamo={r['aamo']}")


## Step 5: Qualitative attention-drift figures


In [ ]:
import generate_full_scale_figures as G
import sys
sys.argv = ["generate_full_scale_figures.py", "--n", "3"]
G.main()


## Step 6: Confirm outputs landed on Drive

Ping Dhinanjaya with `results/baseline_comparison.md` and the `attention_drift_*_full_scale.png` figures.


In [ ]:
import paths
print("outputs under", paths.RESULTS_DIR)
for p in sorted(paths.CKPT_DIR.glob("*.pt")):
    print(" ckpt", p.name, p.stat().st_size)
for p in sorted(paths.RESULTS_DIR.rglob("*")):
    if p.is_file():
        print(" result", p.relative_to(paths.RESULTS_DIR))
